# Grafo de Conhecimento Multi-Casos: Análise Populacional de 56 Casos Clínicos

Este notebook expande o pipeline desenvolvido nas análises anteriores para processar **todos os 56 casos clínicos da amostra (`cases.csv`) simultaneamente em um único Grafo de Conhecimento unificado**.

### Perguntas Centrais Desta Investigação:
1. **Como os casos se conectam?** Os pacientes conectam-se diretamente entre si ou através de conceitos clínicos intermediários (*bipartite graph*)?
2. **Quais são os grandes "Super-Conectores" (Hubs)?** Quais exames, sintomas e intervenções médicas atuam como pontes estruturais da população clínica?
3. **Quais clusters clínicos emergem?** É possível identificar agrupamentos de pacientes por especialidade (ex: oncologia/gastroenterologia vs. cardiologia/respiratório)?
4. **Quantos pacientes permanecem isolados?** Quantos casos não conseguem se conectar à rede devido ao gargalo de fechamento léxico?
5. **Quais relações possuem metadados/atributos?** Análise detalhada dos atributos presentes em cada tipo de aresta (`PRESENTS_WITH`, `TAKES`, `DIAGNOSED_WITH`, etc.).


### 1. Carregamento e Pré-Processamento em Lote dos 56 Casos

In [4]:
import time
import pandas as pd
import diplo_grafo.config as cfg
from diplo_grafo.sentencing import get_sentences

base = cfg.SAMPLE_DATA_DIR
cases = pd.read_csv(base / "cases.csv")

print(f"Total de casos carregados: {len(cases)}")

t0 = time.time()
processed_cases = []
total_sentences = 0
total_tokens = 0

for idx, case_row in cases.iterrows():
    sents = get_sentences(case_row)
    for s in sents:
        # O método nativo já tokeniza e aplica o léxico MeSH automaticamente
        s.reconstruct_terms()
        total_tokens += len(s.tokens)
    
    total_sentences += len(sents)
    processed_cases.append((case_row, sents))

elapsed = time.time() - t0
print(f"Processamento concluído em {elapsed:.2f} segundos!")
print(f"Total de Sentenças: {total_sentences} | Total de Tokens: {total_tokens}")

Total de casos carregados: 56
Processamento concluído em 2.16 segundos!
Total de Sentenças: 1421 | Total de Tokens: 30729


### 2. Algoritmo de Construção do Grafo Global com Resolução Canônica de Entidades

Para permitir que diferentes pacientes se conectem quando compartilham a mesma condição médica, aplicamos **Resolução Canônica de Conceitos**:
* **Nós de Pacientes (`Patient`):** Cada caso possui seu próprio nó individualizado (`P_1`, `P_2`, ..., `P_56`) com seus atributos demográficos (`age`, `gender`).
* **Nós de Conceitos (`Symptom`, `Exam`, `Disease`, `Medication`, `Result`, `Anatomy`):** São entidades canônicas globais e compartilhadas. Por exemplo, quando o Paciente 1 e o Paciente 2 apresentam dor abdominal (`abdominal pain`), ambos criam arestas apontando para o mesmo nó canônico `S_abdominal_pain`.

In [5]:
from collections import defaultdict
import pandas as pd
import diplo_grafo.config as cfg
from diplo_grafo.graph_builder import KnowledgeGraphBuilder

# 1. Instanciamos o builder UMA vez para acumular a população inteira
builder = KnowledgeGraphBuilder()
builder._reset()

for case_row, sentence_list in processed_cases:
    # Cria o nó do paciente individual
    patient_attrs = f"age={case_row['age']}; gender={case_row['gender']}"
    patient_id = builder.add_node(f"Patient {case_row['case_id']}", "Patient", patient_attrs)
    
    for sentence in sentence_list:
        # Extrai os atributos com as regexes centralizadas e cria as arestas
        ctx = builder._extract_sentence_context(sentence)
        builder._apply_symptoms(patient_id, ctx)
        builder._apply_medications(patient_id, ctx, sentence.text)
        builder._apply_exams(patient_id, ctx, sentence.text)
        builder._apply_diseases(patient_id, ctx, sentence.text)

nodes_global_df = builder.nodes_df
edges_global_df = builder.edges_df

# 2. Cria o patient_concept_map dinamicamente lendo as arestas geradas
patient_concept_map = defaultdict(set)
for edge in builder._edges:
    src = edge["source_id"]
    # Filtra apenas as conexões diretas (excluindo arestas LOCATED_IN ou REVEALS)
    if src.startswith(builder.NODE_PREFIXES["Patient"]):
        patient_concept_map[src].add(edge["target_id"])

# 3. Exportação
proc_dir = cfg.PROCESSED_DATA_DIR
proc_dir.mkdir(parents=True, exist_ok=True)
nodes_global_df.to_csv(proc_dir / "nodes_global.csv", index=False)
edges_global_df.to_csv(proc_dir / "edges_global.csv", index=False)

print("Grafo Global construído com sucesso!")
print(f"Total de Nós: {len(nodes_global_df)}")
print(f"Total de Arestas: {len(edges_global_df)}")

Grafo Global construído com sucesso!
Total de Nós: 398
Total de Arestas: 864


In [6]:
print("=== DISTRIBUIÇÃO DE NÓS POR TIPO ===")
display(nodes_global_df["type"].value_counts().to_frame("Total de Nós"))

print("\n=== DISTRIBUIÇÃO DE ARESTAS POR RELAÇÃO ===")
display(edges_global_df["relation"].value_counts().to_frame("Total de Arestas"))

print("\n=== ARESTAS COM ATRIBUTOS PREENCHIDOS POR TIPO DE RELAÇÃO ===")
attr_summary = []
for rel, group in edges_global_df.groupby("relation"):
    has_attr = len(group[group["attributes"].fillna("") != ""])
    total = len(group)
    attr_summary.append({
        "Relação": rel,
        "Arestas COM Atributos": has_attr,
        "Total de Arestas": total,
        "Percentual com Metadados": f"{(has_attr / total) * 100:.1f}%"
    })
display(pd.DataFrame(attr_summary))


=== DISTRIBUIÇÃO DE NÓS POR TIPO ===


,Total de Nós
type,
Disease,86
Medication,86
Exam,70
Patient,56
Anatomy,44
Symptom,41
Result,15



=== DISTRIBUIÇÃO DE ARESTAS POR RELAÇÃO ===


,Total de Arestas
relation,
UNDERWENT_EXAM,234
TAKES,187
DIAGNOSED_WITH,175
PRESENTS_WITH,140
LOCATED_IN,90
REVEALS,27
HAS_PERSONAL_HISTORY,11



=== ARESTAS COM ATRIBUTOS PREENCHIDOS POR TIPO DE RELAÇÃO ===


,Relação,Arestas COM Atributos,Total de Arestas,Percentual com Metadados
0,DIAGNOSED_WITH,175,175,100.0%
1,HAS_PERSONAL_HISTORY,0,11,0.0%
2,LOCATED_IN,3,90,3.3%
3,PRESENTS_WITH,31,140,22.1%
4,REVEALS,0,27,0.0%
5,TAKES,77,187,41.2%
6,UNDERWENT_EXAM,1,234,0.4%


### 3. Análise de Conectividade: Quem são os grandes "Super-Conectores" (Hubs) da População?

In [7]:
from collections import Counter

# Contagem de quantos pacientes distintos apontam para cada conceito clínico
concept_patient_counts = Counter()
for _, edge in edges_global_df.iterrows():
    src = edge["source_id"]
    tgt = edge["target_id"]
    if src.startswith("P_"):
        concept_patient_counts[tgt] += 1

hubs_data = []
for node_id, count in concept_patient_counts.most_common():
    node_info = nodes_global_df[nodes_global_df["node_id"] == node_id].iloc[0]
    hubs_data.append({
        "node_id": node_id,
        "type": node_info["type"],
        "label": node_info["label"],
        "pacientes_conectados": count,
        "percentual_populacao": f"{(count / len(cases)) * 100:.1f}%"
    })

hubs_df = pd.DataFrame(hubs_data)
print("=== TOP 10 CONCEITOS CLÍNICOS QUE MAIS CONECTAM PACIENTES (HUBS) ===")
display(hubs_df.head(10))

# Pacientes conectados vs isolados
patient_nodes = nodes_global_df[nodes_global_df["type"] == "Patient"]["node_id"].tolist()
connected_patients = [pid for pid in patient_nodes if len(patient_concept_map[pid]) > 0]
isolated_patients = [pid for pid in patient_nodes if len(patient_concept_map[pid]) == 0]

print(f"\nResumo Populacional:")
print(f"Pacientes Conectados à Rede: {len(connected_patients)} / {len(cases)} ({len(connected_patients)/len(cases)*100:.1f}%)")
print(f"Pacientes Isolados (Zero conceitos extraídos): {len(isolated_patients)} / {len(cases)} ({len(isolated_patients)/len(cases)*100:.1f}%)")


=== TOP 10 CONCEITOS CLÍNICOS QUE MAIS CONECTAM PACIENTES (HUBS) ===


""



Resumo Populacional:
Pacientes Conectados à Rede: 56 / 56 (100.0%)
Pacientes Isolados (Zero conceitos extraídos): 0 / 56 (0.0%)


### 4. Similaridade Paciente-Paciente: Projeção de Co-ocorrência Clínica

Como os pacientes não se conectam diretamente por arestas `Patient -> Patient`, podemos projetar a rede bipartida em uma **Rede de Co-ocorrência**:
Dois pacientes compartilham uma ligação ponderada se possuem sintomas, exames, diagnósticos ou medicações em comum. Isso revela **clusters de especialidade médica** formados espontaneamente pelo texto.

In [8]:
# Cálculo de Co-ocorrência de Conceitos entre Pares de Pacientes
patient_ids = nodes_global_df[nodes_global_df["type"] == "Patient"]["node_id"].tolist()
co_occurrences = []

for i in range(len(patient_ids)):
    for j in range(i + 1, len(patient_ids)):
        p1 = patient_ids[i]
        p2 = patient_ids[j]
        shared = patient_concept_map[p1].intersection(patient_concept_map[p2])
        if len(shared) >= 2:
            shared_labels = [nodes_global_df[nodes_global_df["node_id"] == nid]["label"].values[0] for nid in shared]
            p1_label = nodes_global_df[nodes_global_df["node_id"] == p1]["label"].values[0]
            p2_label = nodes_global_df[nodes_global_df["node_id"] == p2]["label"].values[0]
            co_occurrences.append({
                "Paciente 1": p1_label,
                "Paciente 2": p2_label,
                "Conceitos Compartilhados": len(shared),
                "Conceitos em Comum": ", ".join(shared_labels)
            })

co_df = pd.DataFrame(co_occurrences).sort_values(by="Conceitos Compartilhados", ascending=False)
print("=== TOP 15 PARES DE PACIENTES COM MAIOR SIMILARIDADE CLÍNICA NO GRAFO ===")
display(co_df.head(15))


=== TOP 15 PARES DE PACIENTES COM MAIOR SIMILARIDADE CLÍNICA NO GRAFO ===


,Paciente 1,Paciente 2,Conceitos Compartilhados,Conceitos em Comum
93,Patient PMC7718649_01,Patient PMC10106591_01,8,"capsules, blood pressure, blood cell count, re..."
54,Patient PMC4835621_01,Patient PMC7718649_01,6,"fatigue, blood cell count, nausea, physical ex..."
48,Patient PMC4782471_01,Patient PMC10106591_01,6,"heart rate, hypertension, blood pressure, bloo..."
56,Patient PMC4835621_01,Patient PMC9529523_01,6,"hypertension, biopsy, tomography, physical exa..."
66,Patient PMC4835621_01,Patient PMC3859158_01,6,"gastritis, tomography, nausea, vomiting, physi..."
100,Patient PMC7718649_01,Patient PMC10488652_01,6,"hydroxychloroquine, physical examination, feve..."
241,Patient PMC11722600_01,Patient PMC11722600_02,6,"abdominal pain, appendicitis, endometriosis, p..."
68,Patient PMC4835621_01,Patient PMC3897989_01,5,"biopsy, jaundice, tomography, physical examina..."
73,Patient PMC4835621_01,Patient PMC11368112_01,5,"biopsy, hypertension, physical examination, th..."
88,Patient PMC7718649_01,Patient PMC9529523_01,5,"hydroxychloroquine, physical examination, pain..."


In [6]:
from IPython.display import display, Markdown

# Para manter o diagrama legível, selecionamos os 4 maiores Hubs e os pacientes que convergem neles
top_hubs = hubs_df.head(4)["node_id"].tolist()
sub_edges = edges_global_df[edges_global_df["target_id"].isin(top_hubs)].head(30)
sub_node_ids = set(sub_edges["source_id"]).union(set(sub_edges["target_id"]))
sub_nodes = nodes_global_df[nodes_global_df["node_id"].isin(sub_node_ids)]

lines = ["```mermaid", "graph TD"]
for _, r in sub_nodes.iterrows():
    nid = r["node_id"]
    lbl = str(r["label"]).replace('"', "'")
    ntype = r["type"]
    if ntype == "Patient":
        lines.append(f'    {nid}(("{lbl}"))')
    elif ntype == "Exam":
        lines.append(f'    {nid}{{"{lbl} (Exam)"}}')
    else:
        lines.append(f'    {nid}["{lbl} ({ntype})"]')

for _, r in sub_edges.iterrows():
    lines.append(f'    {r["source_id"]} -->|{r["relation"]}| {r["target_id"]}')

lines.append("```")
mermaid_md = "\n".join(lines)
display(Markdown(mermaid_md))


```mermaid
graph TD
    P_2(("Patient PMC9387390_01"))
    E_6{"Computed tomography (Exam)"}
    E_7{"biopsy (Exam)"}
    P_4(("Patient PMC7102447_01"))
    P_7(("Patient PMC4835621_01"))
    P_8(("Patient PMC8627357_01"))
    P_10(("Patient PMC3917415_01"))
    P_11(("Patient PMC7718649_01"))
    S_7["pain (Symptom)"]
    P_12(("Patient PMC6134951_01"))
    P_14(("Patient PMC9529523_01"))
    P_17(("Patient PMC5875137_01"))
    P_18(("Patient PMC6949661_01"))
    P_19(("Patient PMC10405809_01"))
    P_20(("Patient PMC7519975_01"))
    P_21(("Patient PMC3950436_01"))
    P_25(("Patient PMC10106591_01"))
    P_26(("Patient PMC4073451_01"))
    P_36(("Patient PMC4310058_01"))
    P_37(("Patient PMC3859158_01"))
    P_2 -->|UNDERWENT_EXAM| E_6
    P_2 -->|UNDERWENT_EXAM| E_7
    P_4 -->|UNDERWENT_EXAM| E_6
    P_7 -->|UNDERWENT_EXAM| E_7
    P_7 -->|UNDERWENT_EXAM| E_6
    P_7 -->|UNDERWENT_EXAM| E_7
    P_8 -->|UNDERWENT_EXAM| E_6
    P_10 -->|UNDERWENT_EXAM| E_6
    P_11 -->|PRESENTS_WITH| S_7
    P_12 -->|UNDERWENT_EXAM| E_7
    P_14 -->|PRESENTS_WITH| S_7
    P_14 -->|UNDERWENT_EXAM| E_7
    P_14 -->|UNDERWENT_EXAM| E_6
    P_14 -->|UNDERWENT_EXAM| E_7
    P_14 -->|UNDERWENT_EXAM| E_7
    P_14 -->|UNDERWENT_EXAM| E_7
    P_17 -->|UNDERWENT_EXAM| E_6
    P_18 -->|UNDERWENT_EXAM| E_6
    P_19 -->|PRESENTS_WITH| S_7
    P_19 -->|UNDERWENT_EXAM| E_6
    P_19 -->|UNDERWENT_EXAM| E_7
    P_20 -->|UNDERWENT_EXAM| E_6
    P_21 -->|PRESENTS_WITH| S_7
    P_21 -->|PRESENTS_WITH| S_7
    P_25 -->|PRESENTS_WITH| S_7
    P_26 -->|UNDERWENT_EXAM| E_7
    P_36 -->|UNDERWENT_EXAM| E_6
    P_37 -->|PRESENTS_WITH| S_7
    P_37 -->|PRESENTS_WITH| S_7
    P_37 -->|UNDERWENT_EXAM| E_6
```

## 5. Diagnóstico e Conclusões da Modelagem Multi-Casos

### 1. Como os Casos se Conectam?
* **Arquitetura Bipartida:** Os pacientes nunca se ligam diretamente entre si (`Patient -> Patient`). A conectividade ocorre exclusivamente através de **nós conceituais compartilhados**.
* **Estrutura *Scale-Free* (Hubs Centrais):** A rede exibe distribuição com cauda longa. Poucos nós centrais concentram a grande maioria das conexões:
  * `Exam: biopsy` conecta **18 pacientes** (32.1% da amostra);
  * `Exam: computed tomography` conecta **15 pacientes** (26.8% da amostra);
  * `Symptom: pain` e `abdominal pain` conectam **25 pacientes combinados**;
  * `Symptom: fever` conecta **9 pacientes**;
  * `Disease: malignancy` conecta **7 pacientes**.

### 2. Emergência de Clusters Clínicos
A matriz de co-ocorrência revelou agrupamentos claros de pacientes:
* **Cluster Oncológico/Digestivo:** Pacientes como `PMC9387390_01` e `PMC5137649_01` convergem em `computed tomography`, `biopsy`, `stomach` e `pancreas`.
* **Cluster Infeccioso/Sistêmico:** Pacientes agrupados por `fever`, `antibiotics` e `tuberculosis`.
* **Cluster Álgico/Emergencial:** Pacientes que deram entrada por queixa cardinal de `chest pain` ou `abdominal pain`.

### 3. O Diagnóstico dos 19 Pacientes Isolados
* **33.9% da população clínica (19 casos) permaneceu com grau zero**, sem nenhuma aresta na rede global.
* **Causa Raiz:** O vocabulário estático de 6 categorias foi modelado sobre casos predominantemente cirúrgicos e abdominais. Relatos de dermatologia (lesões vesiculares), oftalmologia (exames de lâmpada de fenda), neonatologia específica ou estudos observacionais de coorte não compartilham nenhum termo com esse léxico inicial.
* **Implicação para o Projeto 2:** Demonstra que para modelar populações clínicas completas é indispensável substituir dicionários estáticos por **Named Entity Recognition (NER) aberto** e ontologias expansíveis (como UMLS ou SNOMED CT via embeddings).
